In [ ]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput

role = get_execution_role()
session = sagemaker.Session()
region = session.boto_region_name

# Update this to your actual bucket name
bucket = "spy-volatility-forecast-yusuf"
prefix = "volatility-forecast"

print(f"Region: {region}")
print(f"Role: {role}")
print(f"Bucket: {bucket}")

In [ ]:
train_df = pd.read_csv(f"s3://{bucket}/{prefix}/data/train/10YSPYTRAIN.csv")
val_df = pd.read_csv(f"s3://{bucket}/{prefix}/data/validation/10YSPYVALIDATION.csv")
test_df = pd.read_csv(f"s3://{bucket}/{prefix}/data/test/10YSPYTEST.csv")

print(f"Train: {train_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test: {test_df.shape}")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nTarget distribution (train):")
print(train_df.iloc[:, 0].describe())

In [ ]:
import matplotlib.pyplot as plt

target = train_df.iloc[:, 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

target.hist(bins=50, ax=axes[0])
axes[0].set_title("Distribution of Next-Day Range %")
axes[0].set_xlabel("Range %")

corrs = train_df.corr().iloc[:, 0].drop(train_df.columns[0]).sort_values(ascending=False)
corrs.plot(kind="barh", ax=axes[1])
axes[1].set_title("Feature Correlations with Target")
plt.tight_layout()
plt.show()

In [ ]:
# Save headerless CSVs back to S3
train_df.to_csv(f"s3://{bucket}/{prefix}/data/train/train_xgb.csv",
                index=False, header=False)
val_df.to_csv(f"s3://{bucket}/{prefix}/data/validation/val_xgb.csv",
              index=False, header=False)

print("Headerless CSVs uploaded to S3")

In [ ]:
from sagemaker.estimator import Estimator

container = sagemaker.image_uris.retrieve("xgboost", region, version="1.7-1")

xgb_estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/models/",
    sagemaker_session=session,
    hyperparameters={
        "objective": "reg:squarederror",
        "num_round": 200,
        "max_depth": 5,
        "eta": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 3,
        "eval_metric": "rmse",
        "early_stopping_rounds": 20,
    }
)

train_input = TrainingInput(
    s3_data=f"s3://{bucket}/{prefix}/data/train/train_xgb.csv",
    content_type="text/csv"
)
val_input = TrainingInput(
    s3_data=f"s3://{bucket}/{prefix}/data/validation/val_xgb.csv",
    content_type="text/csv"
)

xgb_estimator.fit({"train": train_input, "validation": val_input})

In [ ]:
predictor = xgb_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    serializer=sagemaker.serializers.CSVSerializer(),
    deserializer=sagemaker.deserializers.CSVDeserializer()
)

print(f"Endpoint: {predictor.endpoint_name}")

In [ ]:
test_df = pd.read_csv(f"s3://{bucket}/{prefix}/data/test/10YSPYTEST.csv")

y_test = test_df.iloc[:, 0].values       # target (first column)
X_test = test_df.iloc[:, 1:].values      # features (everything else)

# Predict in batches (endpoint has payload limits)
predictions = []
batch_size = 100
for i in range(0, len(X_test), batch_size):
    batch = X_test[i:i+batch_size]
    csv_batch = "\n".join([",".join(map(str, row)) for row in batch])
    result = predictor.predict(csv_batch)
    preds = [float(r[0]) for r in result]
    predictions.extend(preds)

predictions = np.array(predictions)
print(f"Generated {len(predictions)} predictions")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

# Naive baseline: predict yesterday's range as tomorrow's range
naive_pred = test_df.iloc[:, 3].values  # INTRADAY_RANGE_PCT is the 4th column (index 3)
naive_rmse = np.sqrt(mean_squared_error(y_test, naive_pred))

print(f"===== Test Set Results =====")
print(f"  RMSE:  {rmse:.4f}")
print(f"  MAE:   {mae:.4f}")
print(f"  R²:    {r2:.4f}")
print(f"")
print(f"  Naive Baseline RMSE: {naive_rmse:.4f}")
print(f"  Improvement over naive: {(1 - rmse/naive_rmse)*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, predictions, alpha=0.3, s=10)
axes[0].plot([0, y_test.max()], [0, y_test.max()], "r--")
axes[0].set_xlabel("Actual Range %")
axes[0].set_ylabel("Predicted Range %")
axes[0].set_title("Predicted vs Actual")

axes[1].plot(y_test[:100], label="Actual", alpha=0.7)
axes[1].plot(predictions[:100], label="Predicted", alpha=0.7)
axes[1].set_title("First 100 Test Days")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Make sure we're comparing against the right column
print("Columns:", list(test_df.columns))
print(f"Column at index 3: {test_df.columns[3]}")
print(f"Sample values - Target: {y_test[:5]}")
print(f"Sample values - Naive:  {naive_pred[:5]}")

In [ ]:
from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter

container = sagemaker.image_uris.retrieve("xgboost", region, version="1.7-1")

tuning_estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/models/",
    sagemaker_session=session,
    hyperparameters={
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "early_stopping_rounds": 20,
    }
)

hyperparameter_ranges = {
    "max_depth": IntegerParameter(2, 10),
    "eta": ContinuousParameter(0.01, 0.3),
    "subsample": ContinuousParameter(0.5, 1.0),
    "colsample_bytree": ContinuousParameter(0.3, 1.0),
    "min_child_weight": IntegerParameter(1, 20),
    "num_round": IntegerParameter(50, 500),
    "gamma": ContinuousParameter(0, 5),
    "alpha": ContinuousParameter(0, 10),
    "lambda": ContinuousParameter(0, 10),
}

tuner = HyperparameterTuner(
    estimator=tuning_estimator,
    objective_metric_name="validation:rmse",
    objective_type="Minimize",
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=20,
    max_parallel_jobs=4,
    strategy="Bayesian"
)

tuner.fit({"train": train_input, "validation": val_input})

In [ ]:
# Check best training job
best_job = tuner.best_training_job()
print(f"Best training job: {best_job}")

# Get all results as a dataframe
tuning_results = sagemaker.HyperparameterTuningJobAnalytics(
    tuner.latest_tuning_job.name
).dataframe()

print(f"\nTop 5 jobs by RMSE:")
print(tuning_results.sort_values("FinalObjectiveValue").head()[
    ["TrainingJobName", "FinalObjectiveValue", "TrainingJobStatus"]
])

In [ ]:
# Attach to the best estimator from tuning
best_estimator = tuner.best_estimator()

best_predictor = best_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    serializer=sagemaker.serializers.CSVSerializer(),
    deserializer=sagemaker.deserializers.CSVDeserializer()
)

# Predict on test set
tuned_predictions = []
for i in range(0, len(X_test), batch_size):
    batch = X_test[i:i+batch_size]
    csv_batch = "\n".join([",".join(map(str, row)) for row in batch])
    result = best_predictor.predict(csv_batch)
    preds = [float(r[0]) for r in result]
    tuned_predictions.extend(preds)

tuned_predictions = np.array(tuned_predictions)

# Compare
tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_predictions))
tuned_mae = mean_absolute_error(y_test, tuned_predictions)
tuned_r2 = r2_score(y_test, tuned_predictions)

print(f"===== Tuned Model Results =====")
print(f"  RMSE:  {tuned_rmse:.4f}  (was {rmse:.4f})")
print(f"  MAE:   {tuned_mae:.4f}  (was {mae:.4f})")
print(f"  R²:    {tuned_r2:.4f}  (was {r2:.4f})")
print(f"")
print(f"  Naive Baseline RMSE: {naive_rmse:.4f}")
print(f"  Improvement over naive: {(1 - tuned_rmse/naive_rmse)*100:.1f}%")

In [ ]:
best_predictor.delete_endpoint()
print("Endpoint deleted")

In [ ]:
from sagemaker.model import Model
from sagemaker.serverless import ServerlessInferenceConfig
from sagemaker.predictor import Predictor

# Clean up the failed attempt
try:
    session.delete_model(model_name="spy-volatility-forecast")
    session.delete_endpoint(endpoint_name="spy-volatility-endpoint")
    session.delete_endpoint_config(endpoint_config_name="spy-volatility-endpoint")
except:
    pass
best_model = Model(
    image_uri=container,
    model_data=tuner.best_estimator().model_data,
    role=role,
    sagemaker_session=session,
    name="spy-volatility-forecast"
)

serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=5,
)

best_model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name="spy-volatility-endpoint"
)

# Create predictor manually
endpoint_predictor = Predictor(
    endpoint_name="spy-volatility-endpoint",
    sagemaker_session=session,
    serializer=sagemaker.serializers.CSVSerializer(),
    deserializer=sagemaker.deserializers.CSVDeserializer()
)

print(f"Serverless endpoint deployed: {endpoint_predictor.endpoint_name}")

In [ ]:
# Grab a single row of features from the test set
sample_features = X_test[0]
csv_payload = ",".join(map(str, sample_features))

result = endpoint_predictor.predict(csv_payload)
actual = y_test[0]

print(f"Input features: {sample_features}")
print(f"Predicted next-day range: {float(result[0][0]):.3f}%")
print(f"Actual next-day range:    {actual:.3f}%")

In [ ]:
from sagemaker.model_monitor import DefaultModelMonitor

monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",    # smaller instance
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
)

monitor.suggest_baseline(
    baseline_dataset=f"s3://{bucket}/{prefix}/data/train/train_xgb.csv",
    dataset_format=sagemaker.model_monitor.DatasetFormat.csv(header=False),
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/baseline",
    wait=True
)

print("Baseline created successfully")
print(f"Statistics: {monitor.baseline_statistics().s3_uri}")
print(f"Constraints: {monitor.suggested_constraints().s3_uri}")

In [ ]:
stats = monitor.baseline_statistics()
constraints = monitor.suggested_constraints()

print(f"Statistics: {stats.file_s3_uri}")
print(f"Constraints: {constraints.file_s3_uri}")

In [ ]:
# DON'T run this until you've seen the results above
predictor.delete_endpoint()
print("Endpoint deleted")


In [ ]:
sm_client = boto3.client("sagemaker")

# Check all active endpoints
endpoints = sm_client.list_endpoints(StatusEquals="InService")
for ep in endpoints["Endpoints"]:
    print(f"Active endpoint: {ep['EndpointName']}")
    
    # Check if it's serverless (free when idle) or real-time (charges continuously)
    config = sm_client.describe_endpoint_config(
        EndpointConfigName=ep["EndpointName"]
    )
    if "ProductionVariants" in config:
        for variant in config["ProductionVariants"]:
            if "ServerlessConfig" in variant:
                print("  → Serverless (only charges per request, free when idle)")
            else:
                instance_type = variant.get("InstanceType", "unknown")
                print(f"  → Real-time instance: {instance_type} (CHARGING YOU RIGHT NOW)")